## Agent System Prompts

By default, `create_agent` creates a general-purpose assistant.
A **system prompt** lets you:
* Give the agent a **persona** (e.g., "You are a friendly HR assistant")
* **Constrain scope** (e.g., "Only answer questions about our products")
* Set **output format** (e.g., "Always respond in bullet points")
* Provide **context** the agent needs but the user won't supply every turn

**Topics covered:**
* Passing `SystemMessage` per-call vs. baking it in via `state_modifier`
* Persona assignment
* Scope restriction — refusing off-topic questions
* Formatting instructions
* Dynamic system prompts based on user context

In [ ]:
%run langchain_common.py

## 1. Default agent — no system prompt

Without a system prompt the agent behaves as a generic assistant.

In [ ]:
agent = create_agent(llm_noreason)

result = agent.invoke({"messages": [HumanMessage(content="Who are you and what can you help with?")]})
print(result["messages"][-1].content)

## 2. Method A — `SystemMessage` prepended to the message list

Add a `SystemMessage` as the first element of the `messages` list when calling `invoke`.
This is the **per-call** approach: the system prompt is supplied fresh every turn.
Useful when the system prompt needs to change based on the current user or session.

In [ ]:
CUSTOMER_SUPPORT_PROMPT = """You are Zara, a friendly customer support agent for FlyHigh Airlines.
You help customers with flight bookings, cancellations, and baggage policies.
If a question is not related to FlyHigh Airlines, politely decline and redirect the user.
Keep your answers concise and professional."""

def ask_zara(question: str) -> str:
    """Ask the FlyHigh Airlines agent a question."""
    messages = [
        SystemMessage(content=CUSTOMER_SUPPORT_PROMPT),
        HumanMessage(content=question),
    ]
    result = agent.invoke({"messages": messages})
    return result["messages"][-1].content


print(ask_zara("What is your baggage allowance for economy class?"))
print()
print(ask_zara("Can you recommend a good pizza recipe?"))  # off-topic

## 3. Method B — `state_modifier` baked into the agent

Pass the system prompt directly to `create_agent` via `state_modifier`.
This is the **baked-in** approach: every call automatically includes the system prompt.
No need to prepend it manually — simpler and less error-prone in production.

In [ ]:
HR_PROMPT = """You are Atlas, an HR assistant at TechCorp.
You answer questions about leave policies, benefits, and onboarding procedures.
You do not discuss salaries or performance reviews.
Always respond in a warm, professional tone."""

# state_modifier accepts a string (converted to SystemMessage automatically)
hr_agent = create_agent(llm_noreason, tools=[], state_modifier=HR_PROMPT)

questions = [
    "How many annual leave days do I get?",
    "Can you tell me my colleague's salary?",  # should be declined
    "What is the onboarding process for new hires?",
]

for q in questions:
    result = hr_agent.invoke({"messages": [HumanMessage(content=q)]})
    print(f"Q: {q}")
    print(f"A: {result['messages'][-1].content}")
    print("-" * 60)

## 4. Formatting instructions in the system prompt

You can instruct the agent to format all responses in a specific way —
JSON, markdown, bullet lists, etc.

In [ ]:
JSON_PROMPT = """You are a data extraction assistant.
Always respond with valid JSON only — no prose, no markdown fences.
Extract the requested fields from the user's input."""

json_agent = create_react_agent(llm_noreason, tools=[], state_modifier=JSON_PROMPT)

input_text = "Extract name, age, and city from: John Smith is 34 years old and lives in Karachi."
result = json_agent.invoke({"messages": [HumanMessage(content=input_text)]})
print(result["messages"][-1].content)

In [ ]:
import json

# Parse it back into a Python dict
raw = result["messages"][-1].content
parsed = json.loads(raw)
pp(parsed)

## 5. Dynamic system prompts — personalise per user

The system prompt is just a string. Build it dynamically based on user metadata,
permissions, or preferences — and regenerate it each turn.

In [ ]:
def build_system_prompt(user: dict) -> str:
    """Generate a personalised system prompt for a user."""
    return f"""You are a personalised banking assistant.
The customer's name is {user['name']}.
Their account tier is: {user['tier']}.
Their preferred language is: {user['language']}.
{'They are eligible for premium investment advice.' if user['tier'] == 'platinum' else 'Direct investment queries to a human advisor.'}
Always greet them by name and be appropriately formal for their tier."""


users = [
    {"name": "Ali",   "tier": "standard",  "language": "English"},
    {"name": "Sara",  "tier": "platinum",  "language": "English"},
]

question = "I want to invest my savings — what do you recommend?"

for user in users:
    messages = [
        SystemMessage(content=build_system_prompt(user)),
        HumanMessage(content=question),
    ]
    result = agent.invoke({"messages": messages})
    print(f"=== {user['name']} ({user['tier']}) ===")
    print(result["messages"][-1].content)
    print()

## 6. System prompt + tools + memory — putting it all together

A production agent typically combines all three: a system prompt for behaviour,
tools for capability, and memory for continuity.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field

class OrderInput(BaseModel):
    order_id: str = Field(description="Order ID in the format ORD-XXXX")

ORDERS = {
    "ORD-1001": {"status": "Shipped", "eta": "2025-08-12"},
    "ORD-1002": {"status": "Processing", "eta": "2025-08-15"},
    "ORD-1003": {"status": "Delivered", "eta": "2025-08-01"},
}

@tool(args_schema=OrderInput)
def track_order(order_id: str) -> str:
    """Track the status and estimated delivery date of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        from langchain_core.tools import ToolException
        raise ToolException(f"Order {order_id} not found. Check the order ID and try again.")
    return f"Order {order_id}: {order['status']}. ETA: {order['eta']}"


SHOP_PROMPT = """You are ShopBot, a helpful e-commerce assistant for QuickShop.
You can track orders using the track_order tool.
Only answer questions related to orders and shopping. Be friendly and concise."""

shop_agent = create_react_agent(
    llm_noreason,
    tools=[track_order],
    state_modifier=SHOP_PROMPT,
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "customer-456"}}

turns = [
    "Hi! I placed an order with ID ORD-1001, where is it?",
    "What about ORD-1002?",
    "Can you recommend a good restaurant near me?",  # off-topic
]

for turn in turns:
    result = shop_agent.invoke({"messages": [HumanMessage(content=turn)]}, config)
    print(f"User : {turn}")
    print(f"Agent: {result['messages'][-1].content}")
    print("-" * 60)